In [ ]:
import numpy as np
import pyvista as pv
# pv.set_jupyter_backend('html') # interactive
pv.set_jupyter_backend('static') # static
pv.global_theme.transparent_background = True

render_configs = {
    'ambient': 0.1,          # base color visible even in shadow (0–1)
    'diffuse': 1.,          # main shading from light direction (0–1)
    'specular': 0.6,         # shininess / highlight intensity (0–1)
    'specular_power': 30,    # sharpness of specular highlight (0–128)
    'smooth_shading': True,
}

texture_path = '../../docs/assets/texture-symbol-grid.png'

## Near-isometric datasets

### FAUST_r

In [ ]:
import pandas as pd
from pathlib import Path

atm_path = Path('../../output/faust_r_uncertainty/bench/texture-transfer/full-coverage-soft-canonization/')
atm_path_ = Path('../../../sam3d-for-shape-corr/output/faust_r')
ulrssm_path = Path('../../../sam3d-for-shape-corr/src/fm/experiment/faust_r/ulrssm-20260311-222324/bench/faust_r/')
zoomout_path = Path('../../../sam3d-for-shape-corr/src/fm/experiment/zoom-out-spatial-20260302-194051/bench/faust_r/')

# load benchmark results
our_method = pd.read_csv(atm_path_ / 'geodist.csv')
ulrssm = pd.read_csv(ulrssm_path / 'metric_geodist_per_batch.csv')
zoom_out = pd.read_csv(zoomout_path / 'metric_geodist_per_batch.csv')

df = pd.DataFrame({
    'batch_idx': range(len(our_method)),
    'shape_src': our_method['shape_src'].values,
    'shape_tgt': our_method['shape_tgt'].values,
    'our_method': our_method['geodist'].values,
    'ulrssm': ulrssm['value'].values,
    'zoom_out': zoom_out['value'].values
})

# identify where our method wins
df['is_best'] = (df['our_method'] < df['ulrssm']) & (df['our_method'] < df['zoom_out'])
df['disparity'] = df[['ulrssm', 'zoom_out']].min(axis=1) - df['our_method']

# filter and sort (best & non-self matches)
best_batches = df[df['is_best'] & (df['shape_src'] != df['shape_tgt'])].sort_values('disparity', ascending=False)

print(f"Total batches where our method wins: {len(best_batches)}")
print(f"\nTop 10 batches with largest disparity:")
print(best_batches.head(10).to_string(index=False))
print(f"\n{'='*60}")
print(f"Maximum disparity batch:")
print(best_batches.iloc[0].to_string())

In [ ]:
# import matplotlib.pyplot as plt

# for select in range(5):
#     batch = best_batches.iloc[select]
#     print(batch)
#     name_src, name_tgt = batch['shape_src'], batch['shape_tgt']
#     name_src, name_tgt

#     plt.imshow(plt.imread(atm_path / 'texture-transfer' / 'spectral-icp' / f'{name_src}--{name_tgt}.png'))
#     plt.title(f"ATM")
#     plt.axis('off')
#     plt.show()

#     plt.imshow(plt.imread(ulrssm_path.parent / 'texture-transfer' / f'{name_src}--{name_tgt}.png'))
#     plt.title(f"ULRSSM")
#     plt.axis('off')
#     plt.show()

#     plt.imshow(plt.imread(zoomout_path / 'texture-transfer' / f'{name_src}--{name_tgt}.png'))
#     plt.title(f"ZoomOut")
#     plt.axis('off')
#     plt.show()

In [ ]:
batch = df.iloc[299]
print(batch)
name_src, name_tgt = batch['shape_src'], batch['shape_tgt']

src_mesh = pv.read(ulrssm_path.parent / 'texture-transfer' / f'{name_src}--{name_tgt}' / 'src.obj')
tgt_our = pv.read(atm_path / f'{name_src}--{name_tgt}' / 'tgt.obj')
tgt_ulrssm = pv.read(ulrssm_path.parent / 'texture-transfer' / f'{name_src}--{name_tgt}' / 'tgt.obj')
tgt_zoomout = pv.read(zoomout_path / 'texture-transfer' / f'{name_src}--{name_tgt}' / 'tgt.obj')

shift = np.array([1, 0, 0])

pl = pv.Plotter(window_size=[1920*2, 1080*2])

pl.add_mesh(
    mesh=src_mesh,
    texture=pv.read_texture(texture_path),
    **render_configs,
)
pl.add_mesh(
    mesh=tgt_our.translate(shift),
    texture=pv.read_texture(texture_path),
    **render_configs,
)
pl.add_mesh(
    mesh=tgt_ulrssm.translate(shift*2),
    texture=pv.read_texture(texture_path),
    **render_configs,
)
pl.add_mesh(
    mesh=tgt_zoomout.translate(shift*3),
    texture=pv.read_texture(texture_path),
    **render_configs,
)

pl.enable_parallel_projection()
pl.camera_position = 'xy'
pl.camera.zoom(1.8)
pl.show()

### SCAPE_r

In [ ]:
import pandas as pd
from pathlib import Path

atm_path = Path('../../output/scape_r_uncertainty/bench/texture-transfer/full-coverage-soft-canonization/')
atm_path_ = Path('../../../sam3d-for-shape-corr/output/scape_r')
ulrssm_path = Path('../../../sam3d-for-shape-corr/src/fm/experiment/scape_r/ulrssm-20260312-000407/bench/scape_r')
zoomout_path = Path('../../../sam3d-for-shape-corr/src/fm/experiment/zoom-out-spatial-20260302-194051/bench/scape_r/')

# load benchmark results
our_method = pd.read_csv(atm_path_ / 'geodist.csv')
ulrssm = pd.read_csv(ulrssm_path / 'metric_geodist_per_batch.csv')
zoom_out = pd.read_csv(zoomout_path / 'metric_geodist_per_batch.csv')

df = pd.DataFrame({
    'batch_idx': range(len(our_method)),
    'shape_src': our_method['shape_src'].values,
    'shape_tgt': our_method['shape_tgt'].values,
    'our_method': our_method['geodist'].values,
    'ulrssm': ulrssm['value'].values,
    'zoom_out': zoom_out['value'].values
})

# identify where our method wins
df['is_best'] = (df['our_method'] < df['ulrssm']) & (df['our_method'] < df['zoom_out'])
df['disparity'] = df[['ulrssm', 'zoom_out']].min(axis=1) - df['our_method']

# filter and sort (best & non-self matches)
best_batches = df[df['is_best'] & (df['shape_src'] != df['shape_tgt'])].sort_values('disparity', ascending=False)

print(f"Total batches where our method wins: {len(best_batches)}")
print(f"\nTop 10 batches with largest disparity:")
print(best_batches.head(10).to_string(index=False))
print(f"\n{'='*60}")
print(f"Maximum disparity batch:")
print(best_batches.iloc[0].to_string())

In [ ]:
# import matplotlib.pyplot as plt

# for select in range(5):
#     batch = best_batches.iloc[select]
#     print(batch)
#     name_src, name_tgt = batch['shape_src'], batch['shape_tgt']
#     name_src, name_tgt

#     plt.imshow(plt.imread(atm_path / 'texture-transfer' / 'spectral-icp' / f'{name_src}--{name_tgt}.png'))
#     plt.title(f"ATM")
#     plt.axis('off')
#     plt.show()

#     plt.imshow(plt.imread(ulrssm_path.parent / 'texture-transfer' / f'{name_src}--{name_tgt}.png'))
#     plt.title(f"ULRSSM")
#     plt.axis('off')
#     plt.show()

#     plt.imshow(plt.imread(zoomout_path / 'texture-transfer' / f'{name_src}--{name_tgt}.png'))
#     plt.title(f"ZoomOut")
#     plt.axis('off')
#     plt.show()

In [ ]:
batch = df.iloc[187]
print(batch)
name_src, name_tgt = batch['shape_src'], batch['shape_tgt']

src_mesh = pv.read(ulrssm_path.parent / 'texture-transfer' / f'{name_src}--{name_tgt}' / 'src.obj')
tgt_our = pv.read(atm_path / f'{name_src}--{name_tgt}' / 'tgt.obj')
tgt_ulrssm = pv.read(ulrssm_path.parent / 'texture-transfer' / f'{name_src}--{name_tgt}' / 'tgt.obj')
tgt_zoomout = pv.read(zoomout_path / 'texture-transfer' / f'{name_src}--{name_tgt}' / 'tgt.obj')

shift = np.array([1, 0, 0])

pl = pv.Plotter(window_size=[1920*2, 1080*2])

pl.add_mesh(
    mesh=src_mesh,
    texture=pv.read_texture(texture_path),
    **render_configs,
)
pl.add_mesh(
    mesh=tgt_our.translate(shift),
    texture=pv.read_texture(texture_path),
    **render_configs,
)
pl.add_mesh(
    mesh=tgt_ulrssm.translate(shift*2),
    texture=pv.read_texture(texture_path),
    **render_configs,
)
pl.add_mesh(
    mesh=tgt_zoomout.translate(shift*3),
    texture=pv.read_texture(texture_path),
    **render_configs,
)

pl.enable_parallel_projection()
pl.camera_position = 'xy'
pl.camera.zoom(1.8)
pl.show()

## Non-isometric datasets

### TOPKIDS

In [ ]:
import pandas as pd
from pathlib import Path

atm_path = Path('../../output/topkids_uncertainty/bench/texture-transfer/full-coverage-soft-canonization/')
atm_path_ = Path('../../../sam3d-for-shape-corr/output/topkids')
ulrssm_path = Path('../../../sam3d-for-shape-corr/src/fm/experiment/topkids/ulrssm-20260311-221801/bench/topkids')
zoomout_path = Path('../../../sam3d-for-shape-corr/src/fm/experiment/zoom-out-spatial-20260302-194051/bench/topkids/')

# load benchmark results
our_method = pd.read_csv(atm_path_ / 'geodist.csv')
ulrssm = pd.read_csv(ulrssm_path / 'metric_geodist_per_batch.csv')
zoom_out = pd.read_csv(zoomout_path / 'metric_geodist_per_batch.csv')

df = pd.DataFrame({
    'batch_idx': range(len(our_method)),
    'shape_src': our_method['shape_src'].values,
    'shape_tgt': our_method['shape_tgt'].values,
    'our_method': our_method['geodist'].values,
    'ulrssm': ulrssm['value'].values,
    'zoom_out': zoom_out['value'].values
})

# identify where our method wins
df['is_best'] = (df['our_method'] < df['ulrssm']) & (df['our_method'] < df['zoom_out'])
df['disparity'] = df[['ulrssm', 'zoom_out']].min(axis=1) - df['our_method']

# filter and sort (best & non-self matches)
# best_batches = df[df['is_best'] & (df['shape_src'] != df['shape_tgt'])].sort_values('disparity', ascending=False)
best_batches = df[df['is_best'] & (df['shape_src'] != df['shape_tgt'])].sort_values('our_method', ascending=True)

print(f"Total batches where our method wins: {len(best_batches)}")
print(f"\nTop 10 batches with largest disparity:")
print(best_batches.head(10).to_string(index=False))
print(f"\n{'='*60}")
print(f"Maximum disparity batch:")
print(best_batches.iloc[0].to_string())

In [ ]:
# import matplotlib.pyplot as plt

# for select in range(10):
#     batch = best_batches.iloc[select]
#     print(batch)
#     name_src, name_tgt = batch['shape_src'], batch['shape_tgt']
#     name_src, name_tgt

#     plt.imshow(plt.imread(atm_path / 'texture-transfer' / 'spectral-icp' / f'{name_src}--{name_tgt}.png'))
#     plt.title(f"ATM")
#     plt.axis('off')
#     plt.show()

#     plt.imshow(plt.imread(ulrssm_path.parent / 'texture-transfer' / f'{name_src}--{name_tgt}.png'))
#     plt.title(f"ULRSSM")
#     plt.axis('off')
#     plt.show()

#     plt.imshow(plt.imread(zoomout_path / 'texture-transfer' / f'{name_src}--{name_tgt}.png'))
#     plt.title(f"ZoomOut")
#     plt.axis('off')
#     plt.show()

In [ ]:
batch = df.iloc[8]
print(batch)
name_src, name_tgt = batch['shape_src'], batch['shape_tgt']

src_mesh = pv.read(ulrssm_path.parent / 'texture-transfer' / f'{name_src}--{name_tgt}' / 'src.obj')
tgt_our = pv.read(atm_path / f'{name_src}--{name_tgt}' / 'tgt.obj')
tgt_ulrssm = pv.read(ulrssm_path.parent / 'texture-transfer' / f'{name_src}--{name_tgt}' / 'tgt.obj')
tgt_zoomout = pv.read(zoomout_path / 'texture-transfer' / f'{name_src}--{name_tgt}' / 'tgt.obj')

shift = np.array([0.7, 0, 0])

pl = pv.Plotter(window_size=[1920*2, 1080*2])

pl.add_mesh(
    mesh=src_mesh,
    texture=pv.read_texture(texture_path),
    **render_configs,
)
pl.add_mesh(
    mesh=tgt_our.translate(shift),
    texture=pv.read_texture(texture_path),
    **render_configs,
)
pl.add_mesh(
    mesh=tgt_ulrssm.translate(shift*2),
    texture=pv.read_texture(texture_path),
    **render_configs,
)
pl.add_mesh(
    mesh=tgt_zoomout.translate(shift*3),
    texture=pv.read_texture(texture_path),
    **render_configs,
)

pl.enable_parallel_projection()
pl.camera_position = 'xz'
pl.camera.zoom(1.8)
pl.show()

### SMAL_r

In [ ]:
import pandas as pd
from pathlib import Path

atm_path = Path('../../output/smal_r_prima_uncertainty/bench/texture-transfer/full-coverage-soft-canonization/')
atm_path_ = Path('../../../sam3d-for-shape-corr/output/smal_r')
ulrssm_path = Path('../../../sam3d-for-shape-corr/src/fm/experiment/smal_r/ulrssm-20260311-223517/bench/smal_r')
zoomout_path = Path('../../../sam3d-for-shape-corr/src/fm/experiment/zoom-out-spatial-20260302-194051/bench/smal_r/')

# load benchmark results
our_method = pd.read_csv(atm_path_ / 'geodist.csv')
ulrssm = pd.read_csv(ulrssm_path / 'metric_geodist_per_batch.csv')
zoom_out = pd.read_csv(zoomout_path / 'metric_geodist_per_batch.csv')

df = pd.DataFrame({
    'batch_idx': range(len(our_method)),
    'shape_src': our_method['shape_src'].values,
    'shape_tgt': our_method['shape_tgt'].values,
    'our_method': our_method['geodist'].values,
    'ulrssm': ulrssm['value'].values,
    'zoom_out': zoom_out['value'].values
})

# identify where our method wins
df['is_best'] = (df['our_method'] < df['ulrssm']) & (df['our_method'] < df['zoom_out'])
df['disparity'] = df[['ulrssm', 'zoom_out']].min(axis=1) - df['our_method']

# filter and sort (best & non-self matches)
best_batches = df[df['is_best'] & (df['shape_src'] != df['shape_tgt'])].sort_values('disparity', ascending=False)
# best_batches = df[df['is_best'] & (df['shape_src'] != df['shape_tgt'])].sort_values('our_method', ascending=True)

print(f"Total batches where our method wins: {len(best_batches)}")
print(f"\nTop 10 batches with largest disparity:")
print(best_batches.head(10).to_string(index=False))
print(f"\n{'='*60}")
print(f"Maximum disparity batch:")
print(best_batches.iloc[0].to_string())

In [ ]:
# import matplotlib.pyplot as plt

# for select in range(10):
#     batch = best_batches.iloc[select]
#     print(batch)
#     name_src, name_tgt = batch['shape_src'], batch['shape_tgt']

#     plt.imshow(plt.imread(atm_path / 'texture-transfer' / 'spectral-icp' / f'{name_src}--{name_tgt}.png'))
#     plt.title(f"ATM")
#     plt.axis('off')
#     plt.show()

#     plt.imshow(plt.imread(ulrssm_path.parent / 'texture-transfer' / f'{name_src}--{name_tgt}.png'))
#     plt.title(f"ULRSSM")
#     plt.axis('off')
#     plt.show()

#     plt.imshow(plt.imread(zoomout_path / 'texture-transfer' / f'{name_src}--{name_tgt}.png'))
#     plt.title(f"ZoomOut")
#     plt.axis('off')
#     plt.show()

In [ ]:
batch = df.iloc[293]
print(batch)
name_src, name_tgt = batch['shape_src'], batch['shape_tgt']

src_mesh = pv.read(ulrssm_path.parent / 'texture-transfer' / f'{name_src}--{name_tgt}' / 'src.obj')
tgt_our = pv.read(atm_path / f'{name_src}--{name_tgt}' / 'tgt.obj')
tgt_ulrssm = pv.read(ulrssm_path.parent / 'texture-transfer' / f'{name_src}--{name_tgt}' / 'tgt.obj')
tgt_zoomout = pv.read(zoomout_path / 'texture-transfer' / f'{name_src}--{name_tgt}' / 'tgt.obj')

shift = np.array([0.7, 0, 0])

pl = pv.Plotter(window_size=[1920*2, 1080*2])

pl.add_mesh(
    mesh=src_mesh.rotate_x(180).rotate_y(-40),
    texture=pv.read_texture(texture_path),
    **render_configs,
)
pl.add_mesh(
    mesh=tgt_our.rotate_x(180).rotate_y(-70).translate(shift),
    texture=pv.read_texture(texture_path),
    **render_configs,
)
pl.add_mesh(
    mesh=tgt_ulrssm.rotate_x(180).rotate_y(-70).translate(shift*2),
    texture=pv.read_texture(texture_path),
    **render_configs,
)
pl.add_mesh(
    mesh=tgt_zoomout.rotate_x(180).rotate_y(-70).translate(shift*3),
    texture=pv.read_texture(texture_path),
    **render_configs,
)

pl.enable_parallel_projection()
pl.camera_position = 'xy'
pl.camera.zoom(1.8)
pl.show()